# Using the BaseReducible Module in baseobjects

## Introduction

The `BaseReducible` module provides an abstract class that implements basic functions for object reduction and pickling. It extends `BaseObject` to add functionality for properly handling both `__dict__` and `__slots__` attributes during serialization and deserialization.

This module is particularly important for classes that use `__slots__` for memory optimization or to restrict attribute assignment, as it ensures that both regular attributes and slot attributes are properly preserved during pickling and unpickling operations.

This tutorial covers:
- Understanding the purpose and design of `BaseReducible`
- Using the pickling and unpickling functionality
- Creating custom classes that inherit from `BaseReducible`
- Advanced features and best practices

**Prerequisites:**
- Basic understanding of Python's pickling mechanism
- Familiarity with Python's `__dict__` and `__slots__` attributes
- Knowledge of the `BaseObject` class from the baseobjects package

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [1]:
from baseobjects.bases.basereducible import BaseReducible
import pickle
import pathlib

## Core Functionality

The `BaseReducible` class is an abstract class that extends `BaseObject` to provide standardized methods for object serialization and deserialization through Python's pickle mechanism. It properly handles both regular attributes (stored in `__dict__`) and slot attributes (defined in `__slots__`), ensuring that all object state is preserved during pickling and unpickling.

### Key Features

1. **Proper Pickling Support**: Implements `__getstate__` and `__setstate__` methods to handle pickling and unpickling
2. **Slots Support**: Properly handles both `__dict__` and `__slots__` attributes
3. **Inheritance from BaseObject**: Inherits all the functionality of `BaseObject`, including copying and deep copying

Let's create a simple class that inherits from `BaseReducible`:

In [2]:
class Person(BaseReducible):
    """A simple class representing a person."""

    def __init__(self, name, age) -> None:
        super().__init__()
        self.name = name
        self.age = age

    def __repr__(self) -> str:
        return f"Person(name='{self.name}', age={self.age})"


# Create a Person instance
alice = Person("Alice", 30)
print(alice)

Person(name='Alice', age=30)


### Pickling and Unpickling

One of the key features of `BaseReducible` is its implementation of Python's pickling protocol. Let's see how pickling and unpickling work with `BaseReducible`:

In [3]:
# Pickle the Person instance
pickled_alice = pickle.dumps(alice)
print(f"Pickled data (bytes): {pickled_alice[:30]}... (truncated)")

# Unpickle the Person instance
unpickled_alice = pickle.loads(pickled_alice)
print(f"Unpickled object: {unpickled_alice}")

# Check if the unpickled object is equal to the original
print(f"Are they the same object? {alice is unpickled_alice}")
print("Do they have the same attributes?")
print(f"  - Name: {alice.name == unpickled_alice.name}")
print(f"  - Age: {alice.age == unpickled_alice.age}")

Pickled data (bytes): b'\x80\x04\x956\x00\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\x06Person'... (truncated)
Unpickled object: Person(name='Alice', age=30)
Are they the same object? False
Do they have the same attributes?
  - Name: True
  - Age: True


### Handling Slots

`BaseReducible` properly handles classes that use `__slots__` for memory optimization or to restrict attribute assignment. Let's create a class that uses `__slots__`:

In [4]:
class SlottedPerson(BaseReducible):
    """A class representing a person, using __slots__ for memory optimization."""

    __slots__ = ("age", "name")

    def __init__(self, name, age) -> None:
        super().__init__()
        self.name = name
        self.age = age

    def __repr__(self) -> str:
        return f"SlottedPerson(name='{self.name}', age={self.age})"


# Create a SlottedPerson instance
bob = SlottedPerson("Bob", 25)
print(bob)

# Pickle the SlottedPerson instance
pickled_bob = pickle.dumps(bob)
print(f"Pickled data (bytes): {pickled_bob[:30]}... (truncated)")

# Unpickle the SlottedPerson instance
unpickled_bob = pickle.loads(pickled_bob)
print(f"Unpickled object: {unpickled_bob}")

# Check if the unpickled object is equal to the original
print(f"Are they the same object? {bob is unpickled_bob}")
print("Do they have the same attributes?")
print(f"  - Name: {bob.name == unpickled_bob.name}")
print(f"  - Age: {bob.age == unpickled_bob.age}")

SlottedPerson(name='Bob', age=25)
Pickled data (bytes): b'\x80\x04\x95?\x00\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\rSlotte'... (truncated)
Unpickled object: SlottedPerson(name='Bob', age=25)
Are they the same object? False
Do they have the same attributes?
  - Name: True
  - Age: True


### Handling Both Dict and Slots

`BaseReducible` can also handle classes that use both `__dict__` and `__slots__`:

In [5]:
class HybridPerson(BaseReducible):
    """A class representing a person, using both __dict__ and __slots__."""

    __slots__ = ("age", "name")

    def __init__(self, name, age, **kwargs) -> None:
        super().__init__()
        self.name = name
        self.age = age

        # Add any additional attributes to __dict__
        for key, value in kwargs.items():
            setattr(self, key, value)

    def __repr__(self) -> str:
        dict_attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"HybridPerson(name='{self.name}', age={self.age}{', ' + dict_attrs if dict_attrs else ''})"


# Create a HybridPerson instance with both slot and dict attributes
charlie = HybridPerson("Charlie", 35, occupation="Engineer", city="New York")
print(charlie)

# Pickle the HybridPerson instance
pickled_charlie = pickle.dumps(charlie)
print(f"Pickled data (bytes): {pickled_charlie[:30]}... (truncated)")

# Unpickle the HybridPerson instance
unpickled_charlie = pickle.loads(pickled_charlie)
print(f"Unpickled object: {unpickled_charlie}")

# Check if the unpickled object is equal to the original
print(f"Are they the same object? {charlie is unpickled_charlie}")
print("Do they have the same attributes?")
print(f"  - Name: {charlie.name == unpickled_charlie.name}")
print(f"  - Age: {charlie.age == unpickled_charlie.age}")
print(f"  - Occupation: {charlie.occupation == unpickled_charlie.occupation}")
print(f"  - City: {charlie.city == unpickled_charlie.city}")

HybridPerson(name='Charlie', age=35, occupation='Engineer', city='New York')
Pickled data (bytes): b'\x80\x04\x95n\x00\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\x0cHybrid'... (truncated)
Unpickled object: HybridPerson(name='Charlie', age=35, occupation='Engineer', city='New York')
Are they the same object? False
Do they have the same attributes?
  - Name: True
  - Age: True
  - Occupation: True
  - City: True


## Module Interaction

The `BaseReducible` module interacts with other modules in the baseobjects package, particularly `BaseObject`. These interactions provide enhanced functionality for object serialization and deserialization.

### Interaction with BaseObject

`BaseReducible` inherits from `BaseObject`, which means it also inherits all the functionality of `BaseObject`, including copying and deep copying:

In [6]:
# Create a copy of the Person instance
alice_copy = alice.copy()
print(f"Original: {alice}")
print(f"Copy: {alice_copy}")
print(f"Are they the same object? {alice is alice_copy}")

# Create a deep copy of the Person instance
alice_deepcopy = alice.deepcopy()
print(f"Original: {alice}")
print(f"Deep copy: {alice_deepcopy}")
print(f"Are they the same object? {alice is alice_deepcopy}")

Original: Person(name='Alice', age=30)
Copy: Person(name='Alice', age=30)
Are they the same object? False
Original: Person(name='Alice', age=30)
Deep copy: Person(name='Alice', age=30)
Are they the same object? False


### Interaction with Pickle Module

`BaseReducible` implements the pickling protocol, which means it works seamlessly with Python's `pickle` module:

In [7]:
# Create a list of different types of objects
objects = [
    Person("Dave", 40),
    SlottedPerson("Eve", 45),
    HybridPerson("Frank", 50, hobbies=["Reading", "Hiking"]),
]

# Pickle the list of objects
pickled_objects = pickle.dumps(objects)
print(f"Pickled data (bytes): {pickled_objects[:30]}... (truncated)")

# Unpickle the list of objects
unpickled_objects = pickle.loads(pickled_objects)
print("Unpickled objects:")
for obj in unpickled_objects:
    print(f"  - {obj}")

Pickled data (bytes): b'\x80\x04\x95\xb4\x00\x00\x00\x00\x00\x00\x00]\x94(\x8c\x08__main__\x94\x8c\x06Per'... (truncated)
Unpickled objects:
  - Person(name='Dave', age=40)
  - SlottedPerson(name='Eve', age=45)
  - HybridPerson(name='Frank', age=50, hobbies=['Reading', 'Hiking'])


## Advanced Features

The `BaseReducible` class provides several advanced features that make it powerful for object serialization and deserialization.

### Custom State Management

you can override the `__getstate__` and `__setstate__` methods to customize how your object's state is serialized and deserialized:

In [8]:
class CustomStatePerson(BaseReducible):
    """A class that customizes its state during pickling and unpickling."""

    def __init__(self, name, age, secret) -> None:
        super().__init__()
        self.name = name
        self.age = age
        self.secret = secret

    def __repr__(self) -> str:
        return f"CustomStatePerson(name='{self.name}', age={self.age}, secret='{self.secret}')"

    def __getstate__(self):
        """Customize the state to exclude the secret attribute."""
        state = super().__getstate__()

        # If state is a dictionary, remove the secret attribute
        if isinstance(state, dict):
            state = state.copy()
            if "secret" in state:
                del state["secret"]
        # If state is a tuple with a dictionary as the first element
        elif isinstance(state, tuple) and isinstance(state[0], dict):
            dict_state, slots_state = state
            dict_state = dict_state.copy()
            if "secret" in dict_state:
                del dict_state["secret"]
            state = (dict_state, slots_state)

        return state

    def __setstate__(self, state):
        """Restore the state and set a default value for the secret attribute."""
        super().__setstate__(state)

        # Set a default value for the secret attribute
        if not hasattr(self, "secret"):
            self.secret = "Default secret"


# Create a CustomStatePerson instance
grace = CustomStatePerson("Grace", 55, "Top secret information")
print(f"Original: {grace}")

# Pickle the CustomStatePerson instance
pickled_grace = pickle.dumps(grace)
print(f"Pickled data (bytes): {pickled_grace[:30]}... (truncated)")

# Unpickle the CustomStatePerson instance
unpickled_grace = pickle.loads(pickled_grace)
print(f"Unpickled object: {unpickled_grace}")

# Check if the secret attribute was excluded and then defaulted
print(f"Original secret: {grace.secret}")
print(f"Unpickled secret: {unpickled_grace.secret}")

Original: CustomStatePerson(name='Grace', age=55, secret='Top secret information')
Pickled data (bytes): b'\x80\x04\x95A\x00\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\x11Custom'... (truncated)
Unpickled object: CustomStatePerson(name='Grace', age=55, secret='Default secret')
Original secret: Top secret information
Unpickled secret: Default secret


### Handling Circular References

`BaseReducible` properly handles circular references during pickling and unpickling:

In [9]:
class Node(BaseReducible):
    """A class representing a node in a graph."""

    def __init__(self, name) -> None:
        super().__init__()
        self.name = name
        self.neighbors = []

    def add_neighbor(self, node) -> None:
        self.neighbors.append(node)

    def __repr__(self) -> str:
        return f"Node(name='{self.name}', neighbors={[n.name for n in self.neighbors]})"


# Create some nodes with circular references
node_a = Node("A")
node_b = Node("B")
node_c = Node("C")

# Create circular references
node_a.add_neighbor(node_b)
node_b.add_neighbor(node_c)
node_c.add_neighbor(node_a)  # Circular reference

print(f"Node A: {node_a}")
print(f"Node B: {node_b}")
print(f"Node C: {node_c}")

# Pickle the nodes
pickled_node_a = pickle.dumps(node_a)
print(f"Pickled data (bytes): {pickled_node_a[:30]}... (truncated)")

# Unpickle the nodes
unpickled_node_a = pickle.loads(pickled_node_a)
print(f"Unpickled Node A: {unpickled_node_a}")
print(f"Unpickled Node B: {unpickled_node_a.neighbors[0]}")
print(f"Unpickled Node C: {unpickled_node_a.neighbors[0].neighbors[0]}")

# Verify the circular reference was preserved
print(
    f"Is the circular reference preserved? {unpickled_node_a.neighbors[0].neighbors[0].neighbors[0] is unpickled_node_a}"
)

Node A: Node(name='A', neighbors=['B'])
Node B: Node(name='B', neighbors=['C'])
Node C: Node(name='C', neighbors=['A'])
Pickled data (bytes): b'\x80\x04\x95c\x00\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\x04Node\x94\x93'... (truncated)
Unpickled Node A: Node(name='A', neighbors=['B'])
Unpickled Node B: Node(name='B', neighbors=['C'])
Unpickled Node C: Node(name='C', neighbors=['A'])
Is the circular reference preserved? True


## Examples

Let's explore some practical examples of using the `BaseReducible` module.

### Serializing Complex Objects

`BaseReducible` is particularly useful for serializing complex objects with nested structures:

In [10]:
class Address(BaseReducible):
    """A class representing an address."""

    def __init__(self, street, city, state, zip_code) -> None:
        super().__init__()
        self.street = street
        self.city = city
        self.state = state
        self.zip_code = zip_code

    def __repr__(self) -> str:
        return f"Address(street='{self.street}', city='{self.city}', state='{self.state}', zip_code='{self.zip_code}')"


class Contact(BaseReducible):
    """A class representing a contact with an address."""

    def __init__(self, name, phone, email, address) -> None:
        super().__init__()
        self.name = name
        self.phone = phone
        self.email = email
        self.address = address

    def __repr__(self) -> str:
        return f"Contact(name='{self.name}', phone='{self.phone}', email='{self.email}', address={self.address})"


# Create a complex object with nested structure
address = Address("123 Main St", "Anytown", "CA", "12345")
contact = Contact("John Doe", "555-1234", "john@example.com", address)
print(contact)

# Pickle the complex object
pickled_contact = pickle.dumps(contact)
print(f"Pickled data (bytes): {pickled_contact[:30]}... (truncated)")

# Unpickle the complex object
unpickled_contact = pickle.loads(pickled_contact)
print(f"Unpickled object: {unpickled_contact}")

# Check if the nested structure was preserved
print(f"Is the address a separate object? {unpickled_contact.address is not unpickled_contact}")
print(f"Address street: {unpickled_contact.address.street}")
print(f"Address city: {unpickled_contact.address.city}")

Contact(name='John Doe', phone='555-1234', email='john@example.com', address=Address(street='123 Main St', city='Anytown', state='CA', zip_code='12345'))
Pickled data (bytes): b'\x80\x04\x95\xc8\x00\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\x07Contac'... (truncated)
Unpickled object: Contact(name='John Doe', phone='555-1234', email='john@example.com', address=Address(street='123 Main St', city='Anytown', state='CA', zip_code='12345'))
Is the address a separate object? True
Address street: 123 Main St
Address city: Anytown


### Implementing a Cache with Persistence

`BaseReducible` can be used to implement a cache that can be persisted to disk:

In [11]:
class PersistentCache(BaseReducible):
    """A simple cache that can be persisted to disk."""

    def __init__(self) -> None:
        super().__init__()
        self.cache = {}

    def set(self, key, value) -> None:
        self.cache[key] = value

    def get(self, key, default=None):
        return self.cache.get(key, default)

    def save(self, filename) -> None:
        with pathlib.Path(filename).open("wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, filename):
        with pathlib.Path(filename).open("rb") as f:
            return pickle.load(f)

    def __repr__(self) -> str:
        return f"PersistentCache(items={len(self.cache)})"


# Create a cache and add some items
cache = PersistentCache()
cache.set("key1", "value1")
cache.set("key2", "value2")
cache.set("key3", "value3")

print(f"Cache: {cache}")
print("Cache items:")
for key, value in cache.cache.items():
    print(f"  - {key}: {value}")

# Pickle the cache
pickled_cache = pickle.dumps(cache)
print(f"Pickled data (bytes): {pickled_cache[:30]}... (truncated)")

# Unpickle the cache
unpickled_cache = pickle.loads(pickled_cache)
print(f"Unpickled cache: {unpickled_cache}")
print("Unpickled cache items:")
for key, value in unpickled_cache.cache.items():
    print(f"  - {key}: {value}")

# Note: In a real application, you would use the save() and load() methods
# to persist the cache to disk and load it back.

Cache: PersistentCache(items=3)
Cache items:
  - key1: value1
  - key2: value2
  - key3: value3
Pickled data (bytes): b'\x80\x04\x95c\x00\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\x0fPersis'... (truncated)
Unpickled cache: PersistentCache(items=3)
Unpickled cache items:
  - key1: value1
  - key2: value2
  - key3: value3


## API Highlights

Here are the key components of the `BaseReducible` module API:

### BaseReducible
- `__getstate__()`: Gets the object's state for pickling
- `__setstate__(state)`: Sets the object's state from a pickled state

For more detailed information, consult the full API documentation.

## Troubleshooting / FAQs

### Q: Why use BaseReducible instead of just implementing __getstate__ and __setstate__?

A: `BaseReducible` provides a standardized implementation of `__getstate__` and `__setstate__` that properly handles both `__dict__` and `__slots__` attributes. It also inherits from `BaseObject`, which provides additional functionality like copying and deep copying.

### Q: Can I pickle objects with custom types or functions?

A: Python's pickle module can only serialize certain types of objects. Custom classes can be pickled if they are defined at the module level (not nested inside functions or classes) and don't contain unpicklable objects like file handles, database connections, or lambda functions. `BaseReducible` helps with the serialization process but can't make unpicklable objects picklable.

### Q: How does BaseReducible handle inheritance?

A: `BaseReducible` properly handles inheritance by ensuring that both `__dict__` and `__slots__` attributes from all classes in the inheritance hierarchy are included in the serialized state. When you inherit from `BaseReducible`, you get this behavior automatically.

## Conclusion and Next Steps

In this tutorial, we've explored the `BaseReducible` module and its primary class, `BaseReducible`. We've seen how this class provides proper support for object serialization and deserialization through Python's pickle mechanism, handling both regular attributes and slot attributes.

The `BaseReducible` class is particularly important for classes that use `__slots__` for memory optimization or to restrict attribute assignment, as it ensures that all object state is properly preserved during pickling and unpickling operations.

### Next Steps

- Explore the `BaseObject` module to understand the foundation of `BaseReducible`
- Check out other classes in the baseobjects package that inherit from `BaseReducible`
- Try creating your own custom classes by extending `BaseReducible`
- Consult the full API documentation for more detailed information on the `BaseReducible` module